# Part 1: Bayesian Estimation in the 2PL Item Response Theory (IRT) Model
1. Visualizing the Mechanics (Plotly Script)
Run this block in Google Colab to generate the interactive 2PL Item Response Curves:

In [1]:
import numpy as np
import plotly.graph_objects as go

# Define the 2PL Item Response Function
def p_i(theta, a, b):
    return 1 / (1 + np.exp(-a * (theta - b)))

# Generate a range of latent ability values (theta)
theta_vals = np.linspace(-6, 6, 300)

# Configurations to plot:
# - Two distinct a_i values (a = 0.5 and a = 1.5)
# - For a = 1.5, three difficulty values (b = -2, 0, 2)
curves = [
    {"a": 0.5, "b": 0, "dash": "dash", "name": "a = 0.5, b = 0 (Low Discrimination)"},
    {"a": 1.5, "b": -2, "dash": "solid", "name": "a = 1.5, b = -2 (Easy)"},
    {"a": 1.5, "b": 0, "dash": "solid", "name": "a = 1.5, b = 0 (Moderate)"},
    {"a": 1.5, "b": 2, "dash": "solid", "name": "a = 1.5, b = 2 (Hard)"},
]

fig1 = go.Figure()

for curve in curves:
    p_vals = p_i(theta_vals, curve["a"], curve["b"])
    fig1.add_trace(go.Scatter(
        x=theta_vals,
        y=p_vals,
        mode='lines',
        name=curve["name"],
        line=dict(dash=curve["dash"], width=2.5)
    ))

fig1.update_layout(
    title={
        'text': "Two-Parameter Logistic (2PL) Item Response Curves",
        'y': 0.9, 'x': 0.5, 'xanchor': 'center', 'yanchor': 'top'
    },
    xaxis_title="Latent Ability (θ)",
    yaxis_title="Probability of Correct Response P(Y_i = 1 | θ)",
    xaxis=dict(range=[-6, 6], gridcolor='rgba(0,0,0,0.1)'),
    yaxis=dict(range=[0, 1.05], gridcolor='rgba(0,0,0,0.1)'),
    template="plotly_white",
    legend=dict(yanchor="top", y=0.95, xanchor="left", x=0.05, bgcolor="rgba(255,255,255,0.8)")
)

fig1.show()

### Interpretation of Difficulty ($b_i$) Shifting
The difficulty parameter $b_i$ represents the location along the latent ability axis ($\theta$) where the probability of a correct response is exactly $50\%$ ($P(Y_i = 1 \mid \theta = b_i) = 0.5$).Increasing $b_i$ shifts the entire S-curve horizontally to the right. This means a higher ability level is required for the candidate to have a $50\%$ chance of answering correctly.Decreasing $b_i$ shifts the curve horizontally to the left, representing an easier item where even candidates with lower ability have a high probability of success.

2. Sequential Likelihood Contribution & Joint HistorySingle Response Likelihood ContributionFor a single item response $y_k \in \{0, 1\}$ at step $k$, given latent ability $\theta$:$$L(y_k \mid \theta) = [p_k(\theta)]^{y_k} [1 - p_k(\theta)]^{1 - y_k} = \left( \frac{1}{1 + e^{-a_k(\theta - b_k)}} \right)^{y_k} \left( \frac{e^{-a_k(\theta - b_k)}}{1 + e^{-a_k(\theta - b_k)}} \right)^{1 - y_k}$$Joint Likelihood FunctionAssuming item responses are conditionally independent given $\Theta = \theta$, the joint likelihood for the running history vector $\mathbf{y}^{(k)} = (y_1, y_2, \dots, y_k)$ is the product of individual item likelihoods up to step $k$:$$L(\mathbf{y}^{(k)} \mid \theta) = \prod_{i=1}^{k} L(y_i \mid \theta) = \prod_{i=1}^{k} [p_i(\theta)]^{y_i} [1 - p_i(\theta)]^{1 - y_i}$$3. Mathematical Formulation of the Running UpdateUsing Bayes' Theorem recursively, the posterior density at step $k$, $f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)})$, uses the step $k-1$ posterior as its prior:$$f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)}) \propto L(y_k \mid \theta) \cdot f_{\Theta \mid \mathbf{Y}^{(k-1)}}(\theta \mid \mathbf{y}^{(k-1)})$$Explicitly, up to a normalizing constant:$$f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)}) \propto [p_k(\theta)]^{y_k} [1 - p_k(\theta)]^{1 - y_k} \cdot f_{\Theta \mid \mathbf{Y}^{(k-1)}}(\theta \mid \mathbf{y}^{(k-1)})$$With the base prior given by $f_{\Theta \mid \mathbf{Y}^{(0)}}(\theta) = \frac{1}{\sqrt{2\pi}} \exp\left(-\frac{\theta^2}{2}\right)$.

4. Dynamic Shifting MechanicsWhen a user correctly answers ($y_k = 1$) a difficult item (large $b_k$), the likelihood function $L(y_k = 1 \mid \theta) = p_k(\theta)$ acts as a monotonically increasing logistic function that is close to zero for low values of $\theta$ and rapidly rises toward $1$ near $\theta \approx b_k$.Multiplying the prior $f_{\Theta \mid \mathbf{Y}^{(k-1)}}(\theta \mid \mathbf{y}^{(k-1)})$ by this likelihood heavily penalizes low-ability regions and assigns high weight to $\theta \ge b_k$. As a result, the peak (mode) of the updated posterior density function shifts significantly to the right toward higher ability values.

5. Tracking Certainty and Sharpness (Discrimination $a_k$)The discrimination parameter $a_k$ controls the steepness of the item response curve at $\theta = b_k$ and the magnitude of the log-likelihood gradient (Fisher Information):Large $a_k$ (High Discrimination): The likelihood function transitions sharply from $0$ to $1$ over a narrow range of $\theta$. Multiplying by a sharp likelihood drastically narrows the posterior distribution, rapidly reducing variance and increasing the sharpness (certainty) of the belief state.Small $a_k$ (Low Discrimination): The likelihood function is very flat across $\theta$. It imparts minimal new information, resulting in a minor update to the posterior variance and preserving prior uncertainty.

6. Numerical Implementation of a Running GridBecause the 2PL IRT model lacks a conjugate prior, the posterior must be evaluated on a fixed numerical grid of $\theta$ values (e.g., $M = 1000$ points spanning $[-6, 6]$ with spacing $\Delta\theta$).Sequential Grid Update Algorithm:Initialize: Define a uniform vector $\boldsymbol{\theta} = [\theta_1, \theta_2, \dots, \theta_M]$ and evaluate the initial prior density array $\mathbf{f}^{(0)} = \frac{1}{\sqrt{2\pi}} \exp\left(-\frac{\boldsymbol{\theta}^2}{2}\right)$. Normalize so that $\sum_{j=1}^M f^{(0)}_j \Delta\theta = 1$.For Step $k = 1, \dots, n$:Compute likelihood array $\mathbf{L}_k = [p_k(\theta_j)]^{y_k} [1 - p_k(\theta_j)]^{1 - y_k}$ for all $j \in \{1, \dots, M\}$.Calculate unnormalized posterior array: $\mathbf{f}^*_{k} = \mathbf{L}_k \odot \mathbf{f}_{k-1}$ (element-wise product).Normalize Computational Step: Compute the integral $Z_k = \int_{\mathbb{R}} f^*_k(\theta) d\theta \approx \sum_{j=1}^M f^*_{k, j} \Delta\theta$ (using trapezoidal integration np.trapz).Set normalized state: $\mathbf{f}_k = \frac{\mathbf{f}^*_k}{Z_k}$.

7. Numerical IRT Simulation & Convergence Plot (Colab Script)Run this script in Google Colab to perform the $n=20$ step simulation and render the tracking plot:

In [2]:
import numpy as np
import plotly.graph_objects as go

# Set random seed for reproducibility
np.random.seed(42)

# Parameters
theta_true = 0.75
n_items = 20
theta_grid = np.linspace(-6, 6, 1000)
delta_theta = theta_grid[1] - theta_grid[0]

# 2PL probability function
def p_2pl(theta, a, b):
    return 1.0 / (1.0 + np.exp(-a * (theta - b)))

# Step 0: Initialize with N(0,1) prior grid
current_posterior = (1.0 / np.sqrt(2 * np.pi)) * np.exp(-0.5 * theta_grid**2)
current_posterior /= np.trapz(current_posterior, theta_grid)

# Storage for point estimators over timeline
bayes_estimates = [np.trapz(theta_grid * current_posterior, theta_grid)]
map_estimates = [theta_grid[np.argmax(current_posterior)]]

# Item parameter vectors
a_params = np.random.uniform(0.5, 2.0, size=n_items)
b_params = np.random.normal(0.0, 1.0, size=n_items)

# Running simulation loop
for k in range(n_items):
    a_k = a_params[k]
    b_k = b_params[k]

    # Simulate user response against true ability
    prob_correct = p_2pl(theta_true, a_k, b_k)
    y_k = 1 if np.random.uniform(0, 1) < prob_correct else 0

    # Compute likelihood vector across grid
    p_grid = p_2pl(theta_grid, a_k, b_k)
    likelihood = (p_grid ** y_k) * ((1.0 - p_grid) ** (1 - y_k))

    # Sequential Update and Normalization
    unnormalized_posterior = current_posterior * likelihood
    norm_constant = np.trapz(unnormalized_posterior, theta_grid)
    current_posterior = unnormalized_posterior / norm_constant

    # Extract Point Estimates
    theta_bayes = np.trapz(theta_grid * current_posterior, theta_grid)
    theta_map = theta_grid[np.argmax(current_posterior)]

    bayes_estimates.append(theta_bayes)
    map_estimates.append(theta_map)

# Visualize Progression
steps = list(range(n_items + 1))
fig2 = go.Figure()

# True Ability Reference Line
fig2.add_hline(
    y=theta_true, line_dash="dash", line_color="red", line_width=2,
    annotation_text=f"True Ability (θ_true = {theta_true})", annotation_position="top right"
)

# Estimates
fig2.add_trace(go.Scatter(
    x=steps, y=bayes_estimates, mode='lines+markers',
    name='Posterior Mean (θ̂_Bayes)', line=dict(color='blue', width=2.5)
))

fig2.add_trace(go.Scatter(
    x=steps, y=map_estimates, mode='lines+markers',
    name='MAP Estimate (θ̂_MAP)', line=dict(color='green', width=2, dash='dot')
))

fig2.update_layout(
    title={
        'text': "2PL IRT Bayesian Ability Estimation Timeline (n = 20)",
        'y': 0.92, 'x': 0.5, 'xanchor': 'center', 'yanchor': 'top'
    },
    xaxis_title="Item Step (k)",
    yaxis_title="Estimated Latent Ability (θ̂)",
    template="plotly_white",
    hovermode="x unified",
    legend=dict(yanchor="bottom", y=0.05, xanchor="right", x=0.98)
)

fig2.show()

/tmp/ipykernel_4204/1291185067.py:19: DeprecationWarning:

`trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.

/tmp/ipykernel_4204/1291185067.py:22: DeprecationWarning:

`trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.

/tmp/ipykernel_4204/1291185067.py:44: DeprecationWarning:

`trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.

/tmp/ipykernel_4204/1291185067.py:48: DeprecationWarning:

`trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.



Convergence AnalysisAs the number of observed items $k$ increases, the distance between both estimators ($\hat{\theta}_{\text{Bayes}}^{(k)}$ and $\hat{\theta}_{\text{MAP}}^{(k)}$) and $\theta_{\text{true}} = 0.75$ steadily shrinks.Prior Overcoming: The initial estimates start biased toward $0$ due to the standard normal prior. Each additional item adds log-likelihood information that reduces the influence of the prior.Measurement Confidence: As data accumulates, the posterior distribution becomes progressively narrower around $\theta_{\text{true}}$. The narrowing variance implies that the platform's statistical uncertainty decreases, confirming high estimation confidence.

# Q2: Bayesian Tracking of Click-Through Rates (CTR) via Conjugate Beta-Binomial Updates
## 1. Structural Probability and Properties (Plotly Script)
Run this block in Google Colab to visualize the Beta PDF shape variations:

In [3]:
import numpy as np
import scipy.stats as stats
import plotly.graph_objects as go

theta_grid = np.linspace(0, 1, 500)

beta_configs = [
    {"alpha": 1, "beta": 1, "name": "Uninformative State: Beta(1,1)", "color": "gray", "dash": "dash"},
    {"alpha": 2, "beta": 8, "name": "Right-Skewed State: Beta(2,8)", "color": "blue", "dash": "solid"},
    {"alpha": 8, "beta": 2, "name": "Left-Skewed State: Beta(8,2)", "color": "green", "dash": "solid"}
]

fig3 = go.Figure()

for config in beta_configs:
    pdf_vals = stats.beta.pdf(theta_grid, config["alpha"], config["beta"])
    fig3.add_trace(go.Scatter(
        x=theta_grid, y=pdf_vals, mode='lines',
        name=config["name"],
        line=dict(color=config["color"], dash=config["dash"], width=2.5)
    ))

fig3.update_layout(
    title={
        'text': "Structural Variations of the Beta(α, β) Probability Density Function",
        'y': 0.93, 'x': 0.5, 'xanchor': 'center', 'yanchor': 'top'
    },
    xaxis_title="Conversion Rate Parameter (θ)",
    yaxis_title="Probability Density f(θ)",
    xaxis=dict(range=[0, 1], gridcolor='rgba(0,0,0,0.1)'),
    yaxis=dict(gridcolor='rgba(0,0,0,0.1)'),
    template="plotly_white",
    hovermode="x unified",
    legend=dict(yanchor="top", y=0.95, xanchor="center", x=0.5, bgcolor="rgba(255,255,255,0.7)")
)

fig3.show()

Density Shift InterpretationEqual Parameters ($\alpha = 1, \beta = 1$): Density is uniformly flat over $[0,1]$, representing complete ignorance.$\alpha < \beta$ ($\text{Beta}(2,8)$): Mass shifts heavily toward $0$ (right-skewed), reflecting a belief that low click rates are much more likely.$\alpha > \beta$ ($\text{Beta}(8,2)$): Mass shifts heavily toward $1$ (left-skewed), placing high belief on strong performance.

## 2. Sequential Likelihood and Joint History
Single Impression LikelihoodFor a single outcome $y_k \in \{0, 1\}$ given click rate $\theta$:$$L(y_k \mid \theta) = \theta^{y_k} (1 - \theta)^{1 - y_k}$$Joint History LikelihoodFor impression history vector $\mathbf{y}^{(k)} = (y_1, \dots, y_k)$ with total clicks $C_k = \sum_{i=1}^k y_i$:$$L(\mathbf{y}^{(k)} \mid \theta) = \prod_{i=1}^{k} \theta^{y_i} (1 - \theta)^{1 - y_i} = \theta^{C_k} (1 - \theta)^{k - C_k}$$

## 3. Closed-Form Analytical Updates (Beta-Binomial Conjugacy)
Given prior state $f_{\Theta \mid \mathbf{Y}^{(k-1)}}(\theta) \sim \text{Beta}(\alpha_{k-1}, \beta_{k-1})$:$$f_{\Theta \mid \mathbf{Y}^{(k-1)}}(\theta) \propto \theta^{\alpha_{k-1} - 1} (1 - \theta)^{\beta_{k-1} - 1}$$Applying Bayes' Theorem with new observation $y_k$:$$f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)}) \propto \left[ \theta^{y_k} (1 - \theta)^{1 - y_k} \right] \cdot \left[ \theta^{\alpha_{k-1} - 1} (1 - \theta)^{\beta_{k-1} - 1} \right]$$$$f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)}) \propto \theta^{(\alpha_{k-1} + y_k) - 1} (1 - \theta)^{(\beta_{k-1} + 1 - y_k) - 1}$$Because the functional form matches a Beta kernel, the posterior is analytically proven to remain inside the Beta family with updated shape parameters:$$\alpha_k = \alpha_{k-1} + y_k$$$$\beta_k = \beta_{k-1} + (1 - y_k)$$Posterior Mean Expression$$\mathbb{E}[\Theta \mid \mathbf{Y}^{(k)} = \mathbf{y}^{(k)}] = \frac{\alpha_k}{\alpha_k + \beta_k} = \frac{\alpha_0 + C_k}{(\alpha_0 + \beta_0) + k}$$

##4. Dynamic Shifting Mechanics: Conjugate vs. Non-Conjugate
Observed Click ($y_k = 1$): Increments $\alpha_k = \alpha_{k-1} + 1$, adding power to $\theta$ and shifting the posterior mode toward $1$.Non-Click ($y_k = 0$): Increments $\beta_k = \beta_{k-1} + 1$, adding power to $(1-\theta)$ and pulling the mode toward $0$.Conjugate vs. Non-Conjugate Setup: In the Beta-Binomial conjugate setup, posterior updating reduces to simple integer addition ($\alpha, \beta$ incrementing), completely avoiding numerical integration. In non-conjugate setups (such as the 2PL IRT model), the likelihood-prior product does not match a known standard distribution, requiring numerical grid integration or Markov Chain Monte Carlo (MCMC) techniques at every single step.

##5. Running Point Estimators (Closed-Form Formulas)
For updated parameters $(\alpha_k, \beta_k)$:Running Posterior Mean ($\hat{\theta}_{\text{Bayes}}^{(k)}$):$$\hat{\theta}_{\text{Bayes}}^{(k)} = \frac{\alpha_k}{\alpha_k + \beta_k}$$Running Maximum A Posteriori ($\hat{\theta}_{\text{MAP}}^{(k)}$):$$\hat{\theta}_{\text{MAP}}^{(k)} = \begin{cases} \frac{\alpha_k - 1}{\alpha_k + \beta_k - 2} & \text{if } \alpha_k > 1 \text{ and } \beta_k > 1 \\ 0.0 \text{ or } 1.0 & \text{otherwise} \end{cases}$$
##6. Closed-Form Tracking & Convergence Simulation
Run this final script in Google Colab to perform the $n=100$ impression analytical tracking simulation:

In [4]:
import numpy as np
import scipy.stats as stats
import plotly.graph_objects as go

# Set seed for reproducibility
np.random.seed(42)

# Parameters
theta_true = 0.35
n_impressions = 100
steps = list(range(n_impressions + 1))

# Initial Prior Beta(1,1)
alpha_param = 1
beta_param = 1

running_bayes = [alpha_param / (alpha_param + beta_param)]
running_map = [0.0]

# Sequential Analytical Update Loop
for k in range(1, n_impressions + 1):
    # Simulate user interaction
    y_k = 1 if np.random.uniform(0, 1) < theta_true else 0

    # Exact Closed-Form Updates
    alpha_param += y_k
    beta_param += (1 - y_k)

    # Evaluate exact analytical point estimates
    theta_bayes_k = alpha_param / (alpha_param + beta_param)
    if alpha_param > 1 and beta_param > 1:
        theta_map_k = (alpha_param - 1) / (alpha_param + beta_param - 2)
    else:
        theta_map_k = 0.0 if alpha_param <= beta_param else 1.0

    running_bayes.append(theta_bayes_k)
    running_map.append(theta_map_k)

# Visualize Progression
fig4 = go.Figure()

# True CTR Line
fig4.add_hline(
    y=theta_true, line_dash="dash", line_color="red", line_width=2,
    annotation_text=f"True CTR (θ_true = {theta_true})", annotation_position="bottom right"
)

# Estimates
fig4.add_trace(go.Scatter(
    x=steps, y=running_bayes, mode='lines',
    name='Exact Posterior Mean', line=dict(color='blue', width=2.5)
))

fig4.add_trace(go.Scatter(
    x=steps, y=running_map, mode='lines',
    name='Exact MAP Estimate', line=dict(color='green', width=1.5, dash='dot')
))

fig4.update_layout(
    title={
        'text': "Analytical Beta-Binomial Conjugate Update Timeline (n = 100)",
        'y': 0.93, 'x': 0.5, 'xanchor': 'center', 'yanchor': 'top'
    },
    xaxis_title="Number of User Impressions (k)",
    yaxis_title="Estimated Conversion Rate (θ̂)",
    template="plotly_white",
    hovermode="x unified",
    legend=dict(yanchor="bottom", y=0.05, xanchor="right", x=0.98)
)

fig4.show()

## Analysis of Convergence

As sample size $k \to 100$, the estimators steadily stabilize around $\theta_{\text{true}} = 0.35$.

*  Prior Diminution: The formula $\frac{1 + C_k}{2 + k}
$ shows that as $k$ grows large, the prior constants ($1$ and $2$) become mathematically negligible compared to $C_k$ and $k$.
* Asymptotic Alignment: The Bayesian posterior mean and the MAP estimate converge toward each other and toward the empirical Maximum Likelihood Estimate ($\frac{C_k}{k}$), demonstrating the Bernshteĭn–von Mises theorem in action.